In [1]:
import sys
import re
from pathlib import Path
from collections import defaultdict
from bs4 import BeautifulSoup
import re
from collections import Counter, defaultdict
from typing import List, Tuple

import nltk
from nltk import CFG, PCFG
from nltk.parse import ChartParser, ViterbiParser

# Add utils to path
sys.path.append(str(Path.cwd().parent.parent / 'llm_based_annotation'))
from utils_extraction.html_utils import is_manual_label_tag
from utils_extraction.htmlLabel import HTMLLabel

from utils import extract_parent_level_annotations, get_sublabel_strings, get_ngrams

In [2]:
# Define the three HTML files to analyze
from pathlib import Path
data_dir = Path.cwd().parent.parent / 'data' / 'Documents_Annotés'
final_annotated_dir = Path.cwd().parent.parent / 'data' / 'final' / 'train'

html_files = [
    final_annotated_dir / '1997CanLII16226_ONCA_annotated_EG_revRL_tech.html',
    final_annotated_dir /  '2016NBOMB12_annotated_EG_revRL.html',
    final_annotated_dir / "2019SCC65_annotated_EG_revRL_tech.html",
    final_annotated_dir / "1994CanLII4528NLCA_annotated_GL_revRL.html",
    final_annotated_dir / "2008CSC9_annotated_EG_revRL.html",



]

# Verify files exist
for f in html_files:
    print(f"{'✓' if f.exists() else '✗'} {f.name}")

✓ 1997CanLII16226_ONCA_annotated_EG_revRL_tech.html
✓ 2016NBOMB12_annotated_EG_revRL.html
✓ 2019SCC65_annotated_EG_revRL_tech.html
✓ 1994CanLII4528NLCA_annotated_GL_revRL.html
✓ 2008CSC9_annotated_EG_revRL.html


In [3]:
# Process all files
all_annotations = {
    'decision': [],
    'legislation': [],
    'secondary sources': []
}

for html_file in html_files:
    #print(f"\nProcessing: {html_file.name}")
    
    with open(html_file, 'r', encoding='utf-8') as f:
        html_content = f.read()
    
    annotations = extract_parent_level_annotations(html_content)
    
    # Aggregate results
    for label_type in ['decision', 'legislation', 'secondary sources']:
        count = len(annotations[label_type])
        #print(f"  - {label_type}: {count} annotations")
        all_annotations[label_type].extend(annotations[label_type])

print("\n" + "="*60)
print("TOTAL ANNOTATIONS ACROSS ALL FILES:")
for label_type in ['decision', 'legislation', 'secondary sources']:
    print(f"  {label_type}: {len(all_annotations[label_type])}")
print("="*60)


TOTAL ANNOTATIONS ACROSS ALL FILES:
  decision: 1597
  legislation: 1005
  secondary sources: 241


### FRAGMENT

#### Get the data

In [4]:
decision_fragments = get_sublabel_strings(all_annotations, 'decision', 'fragment', max_items=None)

In [5]:
sec_sources_fragments = get_sublabel_strings(all_annotations, 'secondary sources', 'fragment', max_items=None)

In [6]:
legislation_fragments = get_sublabel_strings(all_annotations, 'legislation', 'fragment', max_items=None)

#### Tokenize them

In [8]:
import re
from typing import List

def normalize(text: str) -> List[str]:
    text = text.lower().strip()

    # --------------------------------------------------
    # 1. Standardize legal units
    # --------------------------------------------------
    text = re.sub(r'\bsections?\b|\bss\.', 'SECTION', text)
    text = re.sub(r's\.', 'SECTION', text)

    text = re.sub(r'\bsubsections?\b', 'SUBSECTION', text)
    text = re.sub(r'\bparagraphs?\b|\bpara\.', 'PARA', text)
    text = re.sub(r'\bsubparagraphs?\b', 'SUBPARA', text)

    text = re.sub(r'\barticles?\b|\barts?\.', 'ARTICLE', text)
    text = re.sub(r'\brules?\b', 'RULE', text)

    text = re.sub(r'\bparts?\b', 'PART', text)
    text = re.sub(r'\bschedules?\b|\bsched\.', 'SCHEDULE', text)
    text = re.sub(r'\bschs\.', 'SCHEDULE', text)

    text = re.sub(r'\bpp\.', 'PP', text)
    text = re.sub(r'\bp\.', 'P', text)
    text = re.sub(r'\bparas\.', 'PARAS', text)
    text = re.sub(r'\bpara\.', 'PARA', text)
    text = re.sub(r'\bparagraph', 'PARA', text)
    text = re.sub(r'\bfootnote', 'FOOTNOTE', text)
    text = re.sub(r'\bpages', 'PAGES', text)
    text = re.sub(r'\bpage', 'PAGE', text)

    # special legal phrases
    text = re.sub(r'\bet seq\.?', 'ETSEQ', text)
    text = re.sub(r'\bthrough\b|\bto\b', 'TO', text)
    text = re.sub(r'\band\b', 'AND', text)
    text = re.sub(r',', ' , ', text)

    # --------------------------------------------------
    # 2. Roman numerals (keep BEFORE NUM)
    # --------------------------------------------------
    text = re.sub(r'\b[ivx]+\b', 'ROMAN', text)

    # --------------------------------------------------
    # 3. Enumerations like (a), (ii)
    # --------------------------------------------------
    text = re.sub(r'\(([a-z]+)\)', '(ALPHA)', text)

    # --------------------------------------------------
    # 4. Hierarchical numeric patterns
    # --------------------------------------------------
    # 638(1)(d) → NUM(PAREN)(PAREN)
    text = re.sub(r'(\d+)\((\d+)\)\(([^)]+)\)', 'NUM(PAREN)(PAREN)', text)

    # 15(1) → NUM(PAREN)
    text = re.sub(r'(\d+)\((\d+)\)', 'NUM(PAREN)', text)

    # 27.09(e) → NUM.NUM(PAREN)
    text = re.sub(r'(\d+)\.(\d+)\(([^)]+)\)', 'NUM.NUM(PAREN)', text)

    # 6.1 → NUM.NUM
    text = re.sub(r'(\d+)\.(\d+)', 'NUM.NUM', text)

    # --------------------------------------------------
    # 5. Ranges (AFTER structure)
    # --------------------------------------------------
    text = re.sub(r'NUM\s*-\s*NUM', 'RANGE', text)
    text = re.sub(r'NUM\s*TO\s*NUM', 'RANGE', text)

    # --------------------------------------------------
    # 6. Standalone numbers
    # --------------------------------------------------
    text = re.sub(r'\d+', 'NUM', text)

    # --------------------------------------------------
    # 7. Cleanup spaces
    # --------------------------------------------------
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = text.split()
    return tokens

In [9]:
tokenized_decision = [normalize(x) for x in decision_fragments]

for raw, tok in zip(decision_fragments, tokenized_decision):
    print(f"{raw:25} → {tok}")

p. 293                    → ['P', 'NUM']
p. 831                    → ['P', 'NUM']
p. 832                    → ['P', 'NUM']
pp. 743-44                → ['PP', 'NUM-NUM']
p. 200                    → ['P', 'NUM']
p. 205                    → ['P', 'NUM']
p.
212                    → ['P', 'NUM']
p.
219                    → ['P', 'NUM']
p. 1006                   → ['P', 'NUM']
p. 485                    → ['P', 'NUM']
pp. 16-17                 → ['PP', 'NUM-NUM']
p. 58                     → ['P', 'NUM']
pp. 222-23                → ['PP', 'NUM-NUM']
pp. 218-32                → ['PP', 'NUM-NUM']
p. 230                    → ['P', 'NUM']
p. 451                    → ['P', 'NUM']
p.
89                     → ['P', 'NUM']
pp. 238-39                → ['PP', 'NUM-NUM']
p. 238                    → ['P', 'NUM']
p. 300                    → ['P', 'NUM']
p. 311                    → ['P', 'NUM']
p. 797                    → ['P', 'NUM']
pp. 26-27                 → ['PP', 'NUM-NUM']
pp. 967-68                →

In [10]:
tokenized_sec_sources = [normalize(x) for x in sec_sources_fragments]

for raw, tok in zip(sec_sources_fragments, tokenized_sec_sources):
    print(f"{raw:25} → {tok}")

p. 193                    → ['P', 'NUM']
pp. 163                   → ['PP', 'NUM']
p. 394                    → ['P', 'NUM']
p. 408                    → ['P', 'NUM']
p. 193                    → ['P', 'NUM']
p. 394                    → ['P', 'NUM']
p. 820                    → ['P', 'NUM']
pp. 819-20                → ['PP', 'NUM-NUM']
pp. 163-65                → ['PP', 'NUM-NUM']
p. 174                    → ['P', 'NUM']
pp. 467-70                → ['PP', 'NUM-NUM']
p. 93                     → ['P', 'NUM']
p. 244                    → ['P', 'NUM']
p. 8                      → ['P', 'NUM']
p. 33                     → ['P', 'NUM']
pp. 541-42                → ['PP', 'NUM-NUM']
pp. 91-93                 → ['PP', 'NUM-NUM']
p. 217                    → ['P', 'NUM']
p. 12-54                  → ['P', 'NUM-NUM']
pp. 459-60                → ['PP', 'NUM-NUM']
p. 220                    → ['P', 'NUM']
p. 134                    → ['P', 'NUM']
p. 286                    → ['P', 'NUM']
p. 139                

In [11]:
tokenized_legi = [normalize(x) for x in legislation_fragments]

for raw, tok in zip(legislation_fragments, tokenized_legi):
    print(f"{raw:25} → {tok}")

s. 8                      → ['SECTION', 'NUM']
ss. 8                     → ['SECTION', 'NUM']
24(2)                     → ['NUM(PAREN)']
s. 1                      → ['SECTION', 'NUM']
ss. 1                     → ['SECTION', 'NUM']
2(b)                      → ['NUM(ALPHA)']
ss. 7                     → ['SECTION', 'NUM']
11(d)                     → ['NUM(ALPHA)']
s. 15(1)                  → ['SECTION', 'NUM(PAREN)']
s. 11(f)                  → ['SECTION', 'NUM(ALPHA)']
s. 11(f)                  → ['SECTION', 'NUM(ALPHA)']
s. 2(a)                   → ['SECTION', 'NUM(ALPHA)']
s. 1                      → ['SECTION', 'NUM']
ss. 1                     → ['SECTION', 'NUM']
2(a)                      → ['NUM(ALPHA)']
s. 8                      → ['SECTION', 'NUM']
s. 24(2)                  → ['SECTION', 'NUM(PAREN)']
ss. 7                     → ['SECTION', 'NUM']
11                        → ['NUM']
15                        → ['NUM']
ss.
2(a)                  → ['SECTION', 'NUM(ALPHA)']
(d)      

#### Count pattern

In [12]:
pattern_counts_decision = Counter(tuple(seq) for seq in tokenized_decision)

print("\nPatterns (Decision Fragments):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_decision.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_decision)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_decision):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(decision_fragments[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Decision Fragments):

[43.19%] ('PARA', 'NUM')
  Example 1: para. 46
  Example 2: para. 55
  Example 3: para. 13

[20.56%] ('P', 'NUM')
  Example 1: p. 293
  Example 2: p. 831
  Example 3: p. 832

[12.77%] ('paraSECTION', 'NUM-NUM')
  Example 1: paras. 34-50
  Example 2: paras. 51-64
  Example 3: paras. 45-46

[7.91%] ('NUM',)
  Example 1: 78
  Example 2: 161
  Example 3: 310

[5.72%] ('PP', 'NUM-NUM')
  Example 1: pp. 743-44
  Example 2: pp. 16-17
  Example 3: pp. 222-23

[4.14%] ('paraSECTION', 'NUM')
  Example 1: paras. 26
  Example 2: paras. 8
  Example 3: paras. 10

[2.19%] ('NUM-NUM',)
  Example 1: 129-31
  Example 2: 26-37
  Example 3: 69-72

[1.58%] ('NUM‑NUM',)
  Example 1: 255‑64
  Example 2: 274‑302
  Example 3: 47‑50

[1.09%] ('paraSECTION', 'NUM‑NUM')
  Example 1: paras. 74‑75
  Example 2: paras. 29‑37
  Example 3: paras. 55‑56

[0.73%] ('PP', 'NUM')
  Example 1: pp. 39
  Example 2: pp. 81
  Example 3: pp. 767

[0.12%] ('PP', 'NUM‑NUM')
  Example 1: pp. 673‑74


In [13]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_decision.items():
    pct = count / len(tokenized_decision)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_decision)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 2
  Total instances in rare patterns: 7
  Outlier mass percentage: 0.85%
  Vocabulary quality score: 99.15%


In [14]:
pattern_counts_sec_sources = Counter(tuple(seq) for seq in tokenized_sec_sources)

print("\nPatterns (Secondary Sources Fragments):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_sec_sources.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_sec_sources)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_sec_sources):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(sec_sources_fragments[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Secondary Sources Fragments):

[59.52%] ('P', 'NUM')
  Example 1: p. 193
  Example 2: p. 394
  Example 3: p. 408

[23.81%] ('PP', 'NUM-NUM')
  Example 1: pp. 819-20
  Example 2: pp. 163-65
  Example 3: pp. 467-70

[2.38%] ('PP', 'NUM')
  Example 1: pp. 163
  Example 2: pp. 342
  Example 3: pp. 204

[2.38%] ('NUM',)
  Example 1: 293
  Example 2: 208
  Example 3: 250

[2.38%] ('PARA', 'NUM.NUM')
  Example 1: para.
13.3
  Example 2: para. 17.224
  Example 3: para. 17.224

[1.59%] ('P', 'NUM-NUM')
  Example 1: p. 12-54
  Example 2: p. 7-19

[1.59%] ('NUM-NUM',)
  Example 1: 369-70
  Example 2: 254-55

[1.59%] ('PP', 'NUM‑NUM')
  Example 1: pp. 118‑19
  Example 2: pp. 531‑32

[0.79%] ('P', 'ROMAN-NUM')
  Example 1: p.
V-12

[0.79%] ('PP', 'NUM-NUM', 'TO', 'NUM-NUM')
  Example 1: pp. 14-3
to 14-6

[0.79%] ('PP', 'NUM‑NUM', 'TO', 'NUM‑NUM')
  Example 1: pp. 7‑5 to 7‑9

[0.79%] ('P', 'NUM‑NUM')
  Example 1: p. 7‑3

[0.79%] ('PARA', 'NUM:NUM')
  Example 1: para. 14:2210

[0.79%] ('PA

In [15]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_sec_sources.items():
    pct = count / len(tokenized_sec_sources)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_sec_sources)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 6
  Total instances in rare patterns: 6
  Outlier mass percentage: 4.76%
  Vocabulary quality score: 95.24%


In [16]:
pattern_counts_legi = Counter(tuple(seq) for seq in tokenized_legi)

print("\nPatterns (Legislation Fragments):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_legi.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_legi)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_legi):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(legislation_fragments[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Legislation Fragments):

[34.99%] ('SECTION', 'NUM')
  Example 1: s. 8
  Example 2: ss. 8
  Example 3: s. 1

[13.71%] ('SECTION', 'NUM(PAREN)(PAREN)')
  Example 1: s. 638(1)(d)
  Example 2: s. 638(1)(b)
  Example 3: s. 3(2)(a)

[13.00%] ('SECTION', 'NUM(PAREN)')
  Example 1: s. 15(1)
  Example 2: s. 24(2)
  Example 3: s. 48(1)

[7.68%] ('SECTION', 'NUM(ALPHA)')
  Example 1: s. 11(f)
  Example 2: s. 11(f)
  Example 3: s. 2(a)

[5.32%] ('NUM',)
  Example 1: 11
  Example 2: 15
  Example 3: 7

[3.19%] ('SECTION', 'NUM(NUM.NUM)')
  Example 1: ss. 97(2.1)
  Example 2: s. 97(2.1)
  Example 3: s. 97(2.1)

[2.36%] ('NUM(ALPHA)',)
  Example 1: 2(b)
  Example 2: 11(d)
  Example 3: 2(a)

[1.77%] ('SECTION', 'NUM.NUM')
  Example 1: section
90.1
  Example 2: section 90.1
  Example 3: section 90.1

[1.42%] ('(ALPHA)',)
  Example 1: (d)
  Example 2: (f)
  Example 3: (g)

[1.42%] ('SUBSECTION', 'NUM(PAREN)')
  Example 1: subsection 73(1)
  Example 2: subsection 60(1)
  Example 3: subsection 

In [17]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_legi.items():
    pct = count / len(tokenized_legi)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_legi)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 48
  Total instances in rare patterns: 83
  Outlier mass percentage: 9.81%
  Vocabulary quality score: 90.19%


#### More

In [18]:
ngram_counts = Counter()

for seq in tokenized_decision:
    for n in range(1, 4):
        ngram_counts.update(get_ngrams(seq, n))

print("\nTop n-grams:")
for k, v in ngram_counts.most_common(10):
    print(v, ":", k)


Top n-grams:
629 : ('NUM',)
355 : ('PARA',)
355 : ('PARA', 'NUM')
170 : ('NUM-NUM',)
169 : ('P',)
169 : ('P', 'NUM')
148 : ('paraSECTION',)
105 : ('paraSECTION', 'NUM-NUM')
54 : ('PP',)
47 : ('PP', 'NUM-NUM')


In [19]:
grammar = CFG.fromstring("""
FRAGMENT -> PREFIXED | SIMPLE

PREFIXED -> PREFIX UNIT
PREFIXED -> PREFIX UNIT LIST
PREFIXED -> PREFIX UNIT ETSEQ

PREFIX -> 'P' | 'PP' | 'PARA' | 'PARAS' | 'FOOTNOTE'

UNIT -> 'NUM' | 'RANGE' | 'NUM-NUM'

LIST -> 'AND' UNIT

SIMPLE -> UNIT
SIMPLE -> UNIT LIST

ETSEQ -> 'ETSEQ'
""")

In [20]:
parser = ChartParser(grammar)

def parse_sequence(seq):
    try:
        trees = list(parser.parse(seq))
        return trees
    except:
        return []

print("\nParsing results:\n")

for seq in tokenized_decision:
    trees = parse_sequence(seq)
    print(seq)
    if trees:
        print("✔ Parsed")
        # print(trees[0])
    else:
        print("✘ Failed")
    print()


Parsing results:

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM-NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['PP', 'NUM']
✔ Parsed

['NUM']
✔ Parsed

['PP', 'NUM']
✔ Parsed

['NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

['NUM']
✔ Parsed

['P', 'NUM']
✔ Parsed

[

In [21]:
productions = []

for seq in tokenized_decision:
    trees = parse_sequence(seq)
    if trees:
        productions += trees[0].productions()


lhs_counts = Counter()
prod_counts = Counter()

for prod in productions:
    lhs_counts[prod.lhs()] += 1
    prod_counts[prod] += 1

pcfg_rules = []

for prod, count in prod_counts.items():
    prob = count / lhs_counts[prod.lhs()]
    pcfg_rules.append(f"{prod} [{prob:.3f}]")


pcfg_grammar = PCFG.fromstring("\n".join(pcfg_rules))

print("\nLearned PCFG:\n")
for rule in pcfg_grammar.productions():
    print(rule)


Learned PCFG:

FRAGMENT -> PREFIXED [0.874]
PREFIXED -> PREFIX UNIT [1.0]
PREFIX -> 'P' [0.293]
UNIT -> 'NUM' [0.902]
PREFIX -> 'PP' [0.092]
UNIT -> 'NUM-NUM' [0.098]
FRAGMENT -> SIMPLE [0.126]
SIMPLE -> UNIT [1.0]
PREFIX -> 'PARA' [0.615]


In [22]:
viterbi_parser = ViterbiParser(pcfg_grammar)

print("\nViterbi parsing:\n")

for seq in tokenized_decision:
    try:
        trees = list(viterbi_parser.parse(seq))
        print(seq)
        print(trees[0])
    except:
        print(seq, "→ Failed")
    print()


Viterbi parsing:

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['PP', 'NUM-NUM']
(FRAGMENT (PREFIXED (PREFIX PP) (UNIT NUM-NUM))) (p=0.00787998)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['PP', 'NUM-NUM']
(FRAGMENT (PREFIXED (PREFIX PP) (UNIT NUM-NUM))) (p=0.00787998)

['P', 'NUM']
(FRAGMENT (PREFIXED (PREFIX P) (UNIT NUM))) (p=0.230986)

['PP', 'NUM-NUM']
(FRAGMENT (PREFIXED (PREFIX PP) (UNIT NUM-NUM))) (p=0.00787998)

['PP', 'NUM-NUM']
(FRA

### CITATION

#### Decision

In [23]:
decision_citation = get_sublabel_strings(all_annotations, 'decision', 'citation', max_items=None)

In [24]:
import re
from typing import List

def normalize_citation(text: str) -> List[str]:
    text = text.strip()
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text)

    # --- ORDINAL series FIRST: (3d), (2d), (4th), (3rd), (2nd) ---
    # Must run before YEAR so "(2d)" isn't mistaken for a year context
    text = re.sub(r'\(\s*\d+(?:st|nd|rd|d|th)\s*\)', '(ORDINAL)', text)

    # --- DOCKET / file numbers (Quebec style): 500-53-000006-984 ---
    text = re.sub(r'\b\d{3}-\d{2}-\d{6,9}-\d{3,4}\b', 'DOCKET', text)

    # --- FULL DATES: "October 18, 1993" / "18 October 1993" ---
    text = re.sub(
        r'\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\s+\d{4}\b',
        'DATE', text
    )
    text = re.sub(
        r'\b(?:January|February|March|April|May|June|July|August|September|'
        r'October|November|December)\s+\d{1,2},\s+\d{4}\b',
        'DATE', text
    )

    # --- BRACKETED YEAR: [1985], [2001] ---
    text = re.sub(r'\[\s*(19|20)\d{2}\s*\]', 'YEAR', text)

    # --- PARENTHETICAL YEAR with comma: (1981), (1944), ---
    # e.g. "(1981), 56 C.C.C." — year is citation date, keep as YEAR
    text = re.sub(r'\(\s*(19|20)\d{2}\s*\)\s*,', 'YEAR ,', text)

    # --- NEUTRAL CITATION TRIBUNALS (before bare year replacement) ---
    # Collapse ALL court codes in neutral citations to TRIBUNAL
    # Covers: SCC, FCA, FC, ONCA, ONSC, BCCA, BCSC, ABCA, QCCA, QCCS,
    #         NBCA, NBQB, CanLII, SKCA, MBCA, NSCA, PECA, etc.
    text = re.sub(
        r'\b(CanLII|[A-Z]{2,4}(?:CA|CS|SC|QB|KB|HCJ|PC)?)\s+(\d+)\b',
        r'TRIBUNAL NUM', text
    )

    # --- BARE YEAR (after bracketed + neutral citation passes) ---
    text = re.sub(r'\b(19|20)\d{2}\b', 'YEAR', text)

    # --- "No." followed by a number ---
    text = re.sub(r'\bno?\.\s*\d+', 'NO NUM', text, flags=re.IGNORECASE)

    # --- COURT / JURISDICTION in parens ---
    # e.g. (C.A.), (S.C.C.), (Ont. Gen. Div.), (Que. C.A.), (Canada PC)
    text = re.sub(
        r'\(\s*(?:'
        r'[A-Z][a-zA-Z.]*\.(?:\s*[A-Z][a-zA-Z.]*\.?)*'   # dotted abbrev
        r'|(?:Gen\.|H\.C\.J\.|Cir\.|Div\.)[\w\s.]*'       # Gen. Div. etc.
        r'|Canada\s+PC'                                     # Canada PC
        r'|[A-Z]{2,5}'                                      # bare caps: SCC
        r')\s*\)',
        '(COURT)', text
    )

    # --- REPORTER abbreviations ---
    # Must cover: O.R., S.C.R., C.C.C., C.R., N.R., D.L.R., W.W.R.,
    #   C.R.R., B.C.L.R., C.P.R., Q.A.C., A.R., C.L.L.C., C.H.R.R.,
    #   C.C.E.L., Alta. L.R., Que. K.B., Imm. L.R., App. Cas.,
    #   U.S., R.R.A., R.J.Q., J.Q., C.S.C.R., S. Ct., etc.
    reporter_pattern = (
        r'\b(?:'
        r'[A-Z](?:\.[A-Z]){1,5}\.?'                         # C.C.C. / S.C.R. / R.J.Q.
        r'|[A-Z][a-z]+\.(?:\s*[A-Z](?:\.[A-Z]+)*\.?)'       # Alta. L.R. / Que. K.B.
        r'|App\.\s*Cas\.'                                     # App. Cas.
        r'|[A-Z]+\s+[A-Z][a-z]+\.'                           # S. Ct.
        r')\b'
    )
    text = re.sub(reporter_pattern, 'REPORTER', text)

    # --- PLACE abbreviations leftover ---
    text = re.sub(r'\b(Hull|Ont|Que|Alta|Man|Sask|Montréal|Montreal)\b\.?', 'PLACE', text)

    # --- Standalone numbers ---
    text = re.sub(r'\b\d+\b', 'NUM', text)

    # --- Collapse whitespace ---
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text.split()

In [25]:
tokenized_decision_citation = [normalize_citation(x) for x in decision_citation]

for raw, tok in zip(decision_citation, tokenized_decision_citation):
    print(f"{raw:25} → {tok}")

33 O.R. (3d) 65           → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
[1997] O.J. No. 1548      → ['YEAR', 'REPORTER.', 'NO', 'NUM']
[1985] 1 S.C.R. 662       → ['TRIBUNAL', 'NUM', 'REPORTER.', 'NUM']
19 C.C.C. (3d) 1          → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
45 C.R. (3d) 289          → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
19 D.L.R. (4th) 314       → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
(1981), 56 C.C.C.
(2d) 193 (Ont. C.A.) → ['YEAR', ',', 'NUM', 'REPORTER.', '(ORDINAL)', 'NUM', '(COURT)']
[1987] 1 S.C.R. 265       → ['TRIBUNAL', 'NUM', 'REPORTER.', 'NUM']
28 C.R.R. 122             → ['NUM', 'REPORTER.', 'NUM']
13
B.C.L.R. (2d) 1        → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
33 C.C.C. (3d) 1          → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
56 C.R. (3d) 193          → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
38 D.L.R. (4th) 508       → ['NUM', 'REPORTER.', '(ORDINAL)', 'NUM']
74
N.R. 276               → ['NUM', 'REPORTER.', 'NUM']
[1987] 3 W.W.R. 699       → ['T

In [26]:
pattern_counts_decision_citation = Counter(tuple(seq) for seq in tokenized_decision_citation)

print("\nPatterns (Decision Citations):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_decision_citation.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_decision_citation)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_decision_citation):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(decision_citation[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Decision Citations):

[40.26%] ('TRIBUNAL', 'NUM', 'REPORTER.', 'NUM')
  Example 1: [1985] 1 S.C.R. 662
  Example 2: [1987] 1 S.C.R. 265
  Example 3: [1987] 3 W.W.R. 699

[25.40%] ('YEAR', 'TRIBUNAL', 'NUM')
  Example 1: 2011 SCC 25
  Example 2: [2008] OJ 1799
  Example 3: [2006] OJ 4615

[16.03%] ('NUM', 'REPORTER.', '(ORDINAL)', 'NUM')
  Example 1: 33 O.R. (3d) 65
  Example 2: 19 C.C.C. (3d) 1
  Example 3: 45 C.R. (3d) 289

[6.73%] ('NUM', 'REPORTER.', 'NUM')
  Example 1: 28 C.R.R. 122
  Example 2: 74
N.R. 276
  Example 3: 36 C.R.R. 193

[1.90%] ('NUM', 'REPORTER.', 'NUM', '(COURT)')
  Example 1: 207 N.R. 321 (S.C.C.)
  Example 2: 18 O.A.C. 321 (C.A.)
  Example 3: 207 N.R. 171 (S.C.C.)

[1.61%] ('NUM', 'REPORTER.', '(ORDINAL)', 'NUM', '(COURT)')
  Example 1: 23 C.R. (4th) 171 (C.A.)
  Example 2: 9 C.R.R. (2d)
196 (Ont. Gen. Div.)
  Example 3: 9 C.R.R. (2d) 220
(Ont. Gen. Div.)

[1.10%] ('YEAR', 'REPORTER.', 'NUM')
  Example 1: [1944]
S.C.R. 267
  Example 2: 2016 D.T.C. 505

In [27]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_decision_citation.items():
    pct = count / len(tokenized_decision_citation)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_decision_citation)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 52
  Total instances in rare patterns: 95
  Outlier mass percentage: 6.95%
  Vocabulary quality score: 93.05%


#### legislation

In [28]:
legislation_citation = get_sublabel_strings(all_annotations, 'legislation', 'citation', max_items=None)

In [29]:
import re
from typing import List


def normalize_legislation_citation(text: str) -> List[str]:
    text = text.strip()

    # --- REGNALE YEARS ---
    text = re.sub(
        r'\b\d{1,2}-\d{1,2}\s+(?:Elizabeth|George|Victoria|Edward|William|Anne|James|Mary|Henry)'
        r'\s+(?:I{1,3}|IV|V|VI|VII|VIII|IX|X|XI)',
        'REGNALE_YEAR',
        text,
    )
    text = re.sub(r'\b(19|20)\d{2}-\d{2}\b', 'YEAR_RANGE', text)

    # --- GAZETTE / DECISION IDENTIFIERS ---
    text = re.sub(r'\bD-\d{4}-\d{3,6}\b', 'DECISION_ID', text)
    text = re.sub(r'\bG\.O\.[A-Z]\.\b', 'GAZETTE', text)
    text = re.sub(r'\b\d{4}\.[IVXLCDM]+\.\d+[A-Z]?\b', 'GAZETTE_REF', text)

    # --- SOR / REGULATION IDENTIFIERS ---
    text = re.sub(r'\bSOR/\d{2,4}-\d{1,4}\b', 'SOR_REG', text)



    text = re.sub(
        r'\bc\.\s*(?:[A-Z][\-\u2011\u2012\u2013]?\d+(?:\.\d+)?|\d+)\b',
        'CHAPTER',
        text,
    )
    text = re.sub(
        r'\bchapter\s+(?:[A-Z][\-\u2011\u2012\u2013]?\d+(?:\.\d+)?|\d+)\b',
        'CHAPTER',
        text,
        flags=re.IGNORECASE,
    )

    # --- PARENTHESISED YEAR: (1985) → YEAR, absorbing the parens ---
    text = re.sub(r'\(\s*(19|20)\d{2}\s*\)', 'YEAR', text)

    # --- DATES ---
    text = re.sub(
        r'\b\d{1,2}\s+(?:January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\s+\d{4}\b',
        'DATE',
        text,
    )
    text = re.sub(r'\b(19|20)\d{2}\b', 'YEAR', text)

    # --- SESSION / LEGISLATURE ---
    text = re.sub(r'\b\d+(?:st|nd|rd|th)\s+Sess\.', 'SESSION', text)
    text = re.sub(r'\b\d+(?:st|nd|rd|th)\s+Leg\.', 'LEGISLATURE', text)

    # --- BILL references ---
    text = re.sub(r'\bBill\s+[A-Z0-9-]+\b', 'BILL', text)

    # --- NO. references ---
    text = re.sub(r'\bNo\.\s*\d+\b', 'NO_NUM', text)

    # --- REPORTER abbreviations ---
    text = re.sub(
        r'\b(?:Que\.|Ont\.|B\.C\.|Alta\.|Man\.|Sask\.|P\.E\.I\.)\s*'
        r'(?:Q\.B\.|K\.B\.|C\.A\.|H\.C\.|H\.C\.J\.)\b',
        'REPORTER',
        text,
    )

    # --- Generic dotted abbreviations (catch-all, runs last among abbrevs) ---
    text = re.sub(r'\b[A-Z](?:\.[A-Z]){1,4}\.?\b', 'ABBREV', text)

    # --- JUDGE names ---
    text = re.sub(
        r"\b[A-Z][a-zA-Z']+(?:\s+[A-Z][a-zA-Z']+)?\s+J\.?\b",
        'JUDGE',
        text,
    )

    # --- PLACE names ---
    text = re.sub(
        r'\b(Hull|Ottawa|Toronto|Montreal|Québec|Vancouver|Canada|Ontario|Quebec)\b',
        'PLACE',
        text,
    )

    # --- Collapse punctuation ---
    text = re.sub(r'\s*,\s*', ' , ', text)
    text = re.sub(r'\s*\(\s*', ' ( ', text)
    text = re.sub(r'\s*\)\s*', ' ) ', text)

    # --- Remaining bare numbers ---
    text = re.sub(r'\b\d+\b', 'NUM', text)

    # --- Clean whitespace ---
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text.split()

In [30]:
tokenized_legislation_citation = [normalize_legislation_citation(x) for x in legislation_citation]

for raw, tok in zip(legislation_citation, tokenized_legislation_citation):
    print(f"{raw:25} → {tok}")

R.S.O.
1990, c. J.3       → ['ABBREV.', 'YEAR', ',', 'c.', 'J.NUM']
R.S.O. 1960, c. 71        → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1970, c. C-34      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985, c. C-46      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985, c. C-46      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985, c.
E-21      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985, c. F-32      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985,
c. I-2       → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.O. 1990, c. J.3       → ['ABBREV.', 'YEAR', ',', 'c.', 'J.NUM']
R.S.C. 1970,
c. L-13      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.O. 1970, c. 261       → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
S.O. 1994, c. 27          → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1970, c. C-34      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.C. 1985, c. C-46      → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.O. 1960, c. 71        → ['ABBREV.', 'YEAR', ',', 'CHAPTER']
R.S.O. 1990, c. C.38      → ['AB

In [33]:
len(tokenized_legislation_citation)

96

In [31]:
pattern_counts_legislation_citation = Counter(tuple(seq) for seq in tokenized_legislation_citation)

print("\nPatterns (Legislation Citations):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_legislation_citation.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_legislation_citation)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_legislation_citation):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(legislation_citation[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Legislation Citations):

[72.92%] ('ABBREV.', 'YEAR', ',', 'CHAPTER')
  Example 1: R.S.O. 1960, c. 71
  Example 2: R.S.C. 1970, c. C-34
  Example 3: R.S.C. 1985, c. C-46

[6.25%] ('Can.', 'ABBREV.', 'YEAR', 'NO_NUM')
  Example 1: Can. T.S. 1974 No. 25
  Example 2: Can. T.S. 1966 No. 29
  Example 3: Can. T.S. 1966 No. 29

[3.12%] ('ABBREV.', 'YEAR', ',', 'c.', 'J.NUM')
  Example 1: R.S.O.
1990, c. J.3
  Example 2: R.S.O. 1990, c. J.3
  Example 3: R.S.O. 1990, c. J.3

[2.08%] ('ABBREV.', 'YEAR', ',', 'c.', 'C.NUM')
  Example 1: R.S.O. 1990, c. C.38
  Example 2: R.S.O. 1990, c. C.38

[2.08%] ('NUM', 'ABBREV.', 'NUM')
  Example 1: 500 U.N.T.S. 223
  Example 2: 500 U.N.T.S. 223

[1.04%] ('ABBREV.', 'YEAR', ',', 'c.', 'M.NUM')
  Example 1: R.S.O. 1990, c. M.3

[1.04%] ('YEAR', ',', 'ABBREV.', 'YEAR', ',', 'CHAPTER')
  Example 1: 1994, S.O.
1994, c. 27

[1.04%] ('Regulation', 'NUM/NUM')
  Example 1: Regulation 372/91

[1.04%] ('CCSM', 'c', 'F175')
  Example 1: CCSM c F175

[1.04%] 

In [34]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_legislation_citation.items():
    pct = count / len(tokenized_legislation_citation)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_legislation_citation)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 0
  Total instances in rare patterns: 0
  Outlier mass percentage: 0.00%
  Vocabulary quality score: 100.00%


#### SOURCE SECONDARY SOURCE

In [35]:
sec_sources_source = get_sublabel_strings(all_annotations, 'secondary sources', 'source', max_items=None)

In [36]:
import re
from typing import List


def normalize_secondary_source(text: str) -> List[str]:

    # ── 0. pre-clean ──────────────────────────────────────────────────────────
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text).strip()

    # ── 1. URLs ───────────────────────────────────────────────────────────────
    text = re.sub(r'on\s*line\s*:\s*\S+', 'URL', text, flags=re.IGNORECASE)
    text = re.sub(r'https?://\S+', 'URL', text)

    # ── 2. ordinals in parens  (2d) (3rd) ────────────────────────────────────
    text = re.sub(r'\(\s*\d+(?:st|nd|rd|d|th)\s*\)', ' ORDINAL ', text)

    # ── 3. edition  ───────────────────────────────────────────────────────────
    text = re.sub(
        r'\(?\s*\d*\s*(?:st|nd|rd|th)?\s*ed(?:ition)?\.?\s*(?:(?:19|20)\d{2})?\s*\)?',
        ' EDITION ', text, flags=re.IGNORECASE
    )
    # bare word like "Second Edition", "Revised Fourth Edition"
    text = re.sub(
        r'\b(?:Revised\s+)?(?:Second|Third|Fourth|Fifth|Sixth|Seventh|'
        r'Eighth|Ninth|Tenth|First|College)\s+Edition\b',
        ' EDITION ', text, flags=re.IGNORECASE
    )

    # ── 4. PUB_INFO  "City: Publisher, YEAR" ─────────────────────────────────
    _CITIES = (
        r'(?:London|Toronto|Ottawa|Montreal|Montréal|Vancouver|'
        r'New\s+York|Chicago|Oxford|Cambridge|Totonto|'
        r'Portland|Kingston|St\.?\s*Paul|Markham|The\s+Hague)'
    )
    text = re.sub(
        r'\(?\s*' + _CITIES + r'(?:\s*,\s*[A-Z][a-z]+\.?)?\s*:\s*[^,]{2,60}?,\s*(?:19|20)\d{2}\s*\)?',
        ' PUB_INFO ', text, flags=re.IGNORECASE
    )

    # ── 5. VOL_ISSUE  "67:3" ─────────────────────────────────────────────────
    text = re.sub(r'\b\d+\s*:\s*\d+\b', ' VOL_ISSUE ', text)

    # ── 6. years ──────────────────────────────────────────────────────────────
    text = re.sub(r'\(?\s*(19|20)\d{2}\s*\)?', ' YEAR ', text)

    # ── 7. split digit prefix glued to letters  62McGill → 62 McGill ─────────
    text = re.sub(r'(?<!\w)(\d+)([A-Za-z])', r'\1 \2', text)

    # ── 8. split digits glued to the RIGHT of a dot  L.J.527 → L.J. 527 ─────
    text = re.sub(r'([A-Za-z]\.)(\d)', r'\1 \2', text)

    # ── 9. JOURNAL_ABBREV ─────────────────────────────────────────────────────
    #   Single big alternation — order longest/most-specific first.
    #   After steps 7-8 all digit gluing is resolved, so we can match cleanly.
    text = re.sub(
        r'\b(?:'
        # --- multi-word named journals ---
        r'Wash\.?\s*&\s*Lee\s+L\.?\s*Rev\.?'
        r'|Can\.?\s+Y\.?B\.?\s+Int\'?l\s+Law'
        r'|Indigenous\s+Law\s+Journal'
        r'|Osgoode\s+Hall\s+L\.?\s*J\.?'
        # --- "School L.J." / "School L. Rev." style ---
        r'|(?:McGill|Dalhousie|Columbia|Colum\.?)\s+L\.?\s*(?:J\.|Rev\.?)'
        r'|Queen\'?s\s+L\.?\s*J\.?'
        r'|Can\.?\s+Bar\s+Rev\.?'
        r'|Can\.?\s+Bus\.?\s+L\.?\s*J\.?'
        # --- pure dotted abbreviations ---
        r'|S\.C\.L\.R\.'
        r'|U\.N\.B\.L\.J\.'
        r'|U\.T\.L\.J\.'
        r'|U\.B\.C\.?\s*L\.?\s*Rev\.?'
        r'|C\.J\.A\.L\.P\.'
        r'|C\.L\.E\.L\.J\.'
        r'|R\.?\s*du\s+B\.'
        r'|Alta\.?\s+L\.?\s*R\.?'
        r'|Is\.?\s*L\.?\s*R\.?'
        r'|Mod\.?\s+L\.?\s*Rev\.?'
        r'|Rev\.?\s+Const\.?\s+Stud\.?'
        r'|Sask\.?\s+L\.?\s*Rev\.?'
        r'|Mich\.?\s+L\.?\s*Rev\.?'
        r'|Adv\.?\s+(?:Q\.?|J\.?)'
        r'|Labour'
        # --- generic fallback: "Xxx L.J." / "Xxx L. Rev." ---
        r'|[A-Z][a-z]+\.?\s+L\.?\s+(?:J\.|Rev\.?)'
        r')\b',
        ' JOURNAL_ABBREV ', text, flags=re.IGNORECASE
    )

    # ── 10. author list (only when followed by ", ed(s).") ───────────────────
    _N = r'(?:[A-Z][a-z]+(?:\s+[A-Z]\.)?(?:\s+[A-Z][a-z]+)?|(?:[A-Z]\.\s*)+[A-Z][a-z]+)'
    text = re.sub(
        r'(?:' + _N + r')(?:\s*,\s*' + _N + r'|\s+and\s+' + _N + r')*'
        r'(?=\s*,\s*eds?\b)',
        ' AUTHOR_LIST ', text, flags=re.IGNORECASE
    )

    # ── 11. editor marker ─────────────────────────────────────────────────────
    text = re.sub(r'\beds?\.(?=,|\s|$)', ' ED_MARKER ', text, flags=re.IGNORECASE)

    # ── 12. book title after ED_MARKER ───────────────────────────────────────
    text = re.sub(
        r'(ED_MARKER\s*,\s*)([A-Za-zÀ-ÿ][^,]*?)(?=\s*(?:YEAR|EDITION|PUB_INFO))',
        r'\1BOOK_TITLE ', text
    )

    # ── 13. residual city / publisher names ───────────────────────────────────
    text = re.sub(
        r'\b(?:London|Toronto|Ottawa|Montreal|Montréal|Vancouver|'
        r'New\s+York|Chicago|Oxford|Cambridge|Totonto)\b',
        ' PUB_PLACE ', text, flags=re.IGNORECASE
    )
    text = re.sub(
        r'\b(?:Macmillan|Emond|Thomson\s+Reuters|LexisNexis|Irwin\s+Law|'
        r'Carswell|Butterworths?|Oxford\s+University\s+Press|OUP|'
        r'Cambridge\s+University\s+Press|Hart|Kluwer)\b',
        ' PUBLISHER ', text, flags=re.IGNORECASE
    )

    # ── 14. institutions ──────────────────────────────────────────────────────
    text = re.sub(r'\bLaw\s+Reform\s+Commission\s+of\s+\w+\b', ' INSTITUTION ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bSupreme\s+Court(?:\s+(?:Law\s+)?of\s+Canada)?\b', ' INSTITUTION ', text, flags=re.IGNORECASE)

    # ── 15. volume / issue / month ────────────────────────────────────────────
    text = re.sub(r'\bvol(?:ume)?\.?\s*\d*', ' VOLUME ', text, flags=re.IGNORECASE)
    text = re.sub(r'\bissue\b', ' ISSUE ', text, flags=re.IGNORECASE)
    text = re.sub(
        r'\b(?:January|February|March|April|May|June|July|August|'
        r'September|October|November|December)\b',
        ' MONTH ', text
    )

    # ── 16. remaining dotted abbreviations → ABBREV ───────────────────────────
    text = re.sub(r'\b[A-Z](?:\.[A-Za-z]){1,6}\.?\b', ' ABBREV ', text)

    # ── 17. bare numbers → NUM ────────────────────────────────────────────────
    text = re.sub(r'\b\d+\b', ' NUM ', text)

    # ── 18. punctuation normalise ─────────────────────────────────────────────
    text = re.sub(r'\s*,\s*', ' , ', text)
    text = re.sub(r'\s*:\s*', ' : ', text)
    text = re.sub(r'\s*\(\s*', ' ( ', text)
    text = re.sub(r'\s*\)\s*', ' ) ', text)

    # ── 19. collapse whitespace ───────────────────────────────────────────────
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text.split()

In [37]:
tokenized_sec_sources_source = [normalize_secondary_source(x) for x in sec_sources_source]

for raw, tok in zip(sec_sources_source, tokenized_sec_sources_source):
    print(f"{raw:25} → {tok}")

10th ed.
(1959)           → ['EDITION', 'YEAR']
(1964), Can. Y.B. Int'l Law 107 → ['YEAR', ',', 'JOURNAL_ABBREV', 'NUM']
38 McGill L.J. 147        → ['NUM', 'McGill', 'ABBREV', '.', 'NUM']
(1992), 15 Dalhousie L.J. 380 → ['YEAR', ',', 'NUM', 'Dalhousie', 'ABBREV', '.', 'NUM']
Studies on the Jury, Law Reform
Commission of Canada (1979) → ['Studies', 'on', 'the', 'Jury', ',', 'INSTITUTION', 'YEAR']
10th ed.
(1959)           → ['EDITION', 'YEAR']
(1992), 15 Dalhousie
L.J. 380 → ['YEAR', ',', 'NUM', 'Dalhousie', 'ABBREV', '.', 'NUM']
(1988-89), 39 Case Western L. Rev. 807 → ['YEAR', '-', 'NUM', ')', ',', 'NUM', 'Case', 'JOURNAL_ABBREV', '.', 'NUM']
(1993), 38 McGill L.J. 147 → ['YEAR', ',', 'NUM', 'McGill', 'ABBREV', '.', 'NUM']
Montreal: Dictionnaires Le Robert, 1988 → ['PUB_INFO']
Glasgow: Geddes and Grosset, 2010 → ['Glasgow', ':', 'G', 'EDITION', 'des', 'and', 'Grosset', ',', 'YEAR']
(1983), 43R. du B.277     → ['YEAR', ',', 'NUM', 'R.', 'du', 'B.', 'NUM']
(1986), 21Is.L.R.269      → [

In [38]:
pattern_counts_sec_source_source = Counter(tuple(seq) for seq in tokenized_sec_sources_source)

print("\nPatterns (Secondary Sources - Source Field):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_sec_source_source.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_sec_sources_source)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_sec_sources_source):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(sec_sources_source[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Secondary Sources - Source Field):

[22.70%] ('YEAR', ',', 'NUM', 'ABBREV', '.', 'NUM')
  Example 1: (2017), 68U.N.B.L.J.87
  Example 2: (2014), 64U.T.L.J.458
  Example 3: (2017), 30C.J.A.L.P.307

[17.18%] ('YEAR', ',', 'NUM', 'JOURNAL_ABBREV', '.', 'NUM')
  Example 1: (1986), 21Is.L.R.269
  Example 2: (2018), 56Alta. L.R.119
  Example 3: (2011), 74Mod. L. Rev.694

[8.59%] ('EDITION',)
  Example 1: 6th ed. 2014
  Example 2: 5th ed. 1995
  Example 3: (2nd ed. 1983)

[4.91%] ('PUB_INFO',)
  Example 1: Montreal: Dictionnaires Le Robert, 1988
  Example 2: London: Allen Lane, 2010
  Example 3: Cambridge: Cambridge University Press, 2008

[4.29%] ('EDITION', 'PUB_INFO')
  Example 1: 5th ed. Oxford: Clarendon Press, 1998
  Example 2: 10th ed. London: Macmillan, 1959
  Example 3: 2nd ed. Markham, Ont.: LexisNexis, 2015

[4.29%] ('YEAR', ',', 'NUM', 'ABBREV', '.', 'ORDINAL', 'NUM')
  Example 1: (2014), 66S.C.L.R.(2d) 233
  Example 2: (2014), 66S.C.L.R.(2d) 1
  Example 3: (2016), 74S.

In [39]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_sec_source_source.items():
    pct = count / len(tokenized_sec_sources_source)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_sec_sources_source)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 29
  Total instances in rare patterns: 29
  Outlier mass percentage: 17.79%
  Vocabulary quality score: 82.21%


### DECISION TITLE

In [40]:
decision_title = get_sublabel_strings(all_annotations, 'decision', 'title', max_items=None)

In [41]:
import re
from typing import List


def normalize_decision_title(text: str) -> List[str]:

    # ------------------------------------------------------------------ #
    # 0. Pre-clean: collapse newlines / extra whitespace                  #
    # ------------------------------------------------------------------ #
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text).strip()

    # ------------------------------------------------------------------ #
    # 1. Fix glued "v." / "c." (no spaces around them)                   #
    #    "Poirierv.Ville" → "Poirier v. Ville"                           #
    # ------------------------------------------------------------------ #
    text = re.sub(r'([A-Za-zÀ-ÿ])(v\.|c\.)([A-Za-zÀ-ÿ])', r'\1 \2 \3', text)

    # ------------------------------------------------------------------ #
    # 2. Year: (YYYY) → YEAR                                             #
    # ------------------------------------------------------------------ #
    text = re.sub(r'\(\s*(19|20)\d{2}\s*\)', 'YEAR', text)

    # ------------------------------------------------------------------ #
    # 3. Case separator "v." / "c." → V                                  #
    # ------------------------------------------------------------------ #
    text = re.sub(r'\bv\.', 'V', text)
    text = re.sub(r'\bc\.', 'V', text)

    # ------------------------------------------------------------------ #
    # 4. Semicolon between joined cases → CASE_SEP                       #
    # ------------------------------------------------------------------ #
    text = re.sub(r'\s*;\s*', ' CASE_SEP ', text)

    # ------------------------------------------------------------------ #
    # 5. Tokenize: split on structural tokens while preserving parens     #
    #    At this point text contains: words, V, YEAR, CASE_SEP, ( )      #
    #    Split into chunks separated by V / YEAR / CASE_SEP / ( / )      #
    #    Each chunk of plain words → PARTY                               #
    # ------------------------------------------------------------------ #
    # Insert spaces around parens so they split cleanly
    text = re.sub(r'\(', ' ( ', text)
    text = re.sub(r'\)', ' ) ', text)

    # Collapse whitespace
    text = re.sub(r'[ \t]+', ' ', text).strip()

    # Now walk tokens: any run of plain words becomes PARTY
    raw_tokens = text.split()
    structural = {'V', 'YEAR', 'CASE_SEP', '(', ')'}
    
    result = []
    party_buf = []

    def flush_party():
        if party_buf:
            result.append('PARTY')
            party_buf.clear()

    for tok in raw_tokens:
        if tok in structural:
            flush_party()
            result.append(tok)
        else:
            # Strip trailing punctuation like commas, periods
            tok_clean = re.sub(r'^[,\.]+|[,\.]+$', '', tok)
            if tok_clean:
                party_buf.append(tok_clean)

    flush_party()

    return result

In [42]:
tokenized_decision_title = [normalize_decision_title(x) for x in decision_title]

for raw, tok in zip(decision_title, tokenized_decision_title):
    print(f"{raw:25} → {tok}")

R. v. Church of Scientology → ['PARTY', 'V', 'PARTY']
Canadian Dredge & Dock Co. v. R. → ['PARTY', 'V', 'PARTY']
R. v. Collins             → ['PARTY', 'V', 'PARTY']
R. v. Carosella (1997)    → ['PARTY', 'V', 'PARTY', 'YEAR']
Andrews
v. Law Society of British Columbia → ['PARTY', 'V', 'PARTY']
Irwin Toy Ltd. v. Quebec (Attorney General) → ['PARTY', 'V', 'PARTY', '(', 'PARTY', ')']
Philippines (Republic) v. Pacificador (1993) → ['PARTY', '(', 'PARTY', ')', 'V', 'PARTY', 'YEAR']
R. v.
Big M Drug Mart Ltd. → ['PARTY', 'V', 'PARTY']
R. v. Church of Scientology (1992) → ['PARTY', 'V', 'PARTY', 'YEAR']
R. v. Church of Scientology (1992) → ['PARTY', 'V', 'PARTY', 'YEAR']
R. v. Church of Scientology (1992) → ['PARTY', 'V', 'PARTY', 'YEAR']
R. v. Church of Scientology; R. v. Zaharia (1987) → ['PARTY', 'V', 'PARTY', 'CASE_SEP', 'PARTY', 'V', 'PARTY', 'YEAR']
R. v. Edwards Books &
Art Ltd. → ['PARTY', 'V', 'PARTY']
R. v. Goldhart            → ['PARTY', 'V', 'PARTY']
R. v. Morgentaler         → ['P

In [43]:
pattern_counts_decision_title = Counter(tuple(seq) for seq in tokenized_decision_title)

print("\nPatterns (Decision Titles):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_decision_title.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_decision_title)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_decision_title):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(decision_title[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Decision Titles):

[44.06%] ('PARTY',)
  Example 1: Vetrovec
  Example 2: The Rhône
  Example 3: Canadian Dredge & Dock

[28.55%] ('PARTY', 'V', 'PARTY')
  Example 1: R. v. Church of Scientology
  Example 2: Canadian Dredge & Dock Co. v. R.
  Example 3: R. v. Collins

[13.98%] ('PARTY', 'V', 'PARTY', '(', 'PARTY', ')')
  Example 1: Irwin Toy Ltd. v. Quebec (Attorney General)
  Example 2: R. v. S. (R.J.)
  Example 3: R. v. E. (A.W.)

[5.54%] ('PARTY', '(', 'PARTY', ')', 'V', 'PARTY')
  Example 1: Alberta (Information and Privacy Commissioner) v. Alberta Te...
  Example 2: Canada (Citizenship and Immigration) v. Khosa
  Example 3: Canada (Attorney General) v. Bedford

[1.82%] ('PARTY', '(', 'PARTY', ')', 'V', 'PARTY', '(', 'PARTY', ')')
  Example 1: Rhône (The) v. Peter A.B. Widener (The)
  Example 2: Rhône (The) v. Peter A.B. Widener
(The)
  Example 3: Rhône (The) v. Peter A.B Widener (The)

[1.24%] ('PARTY', 'V', 'PARTY', 'YEAR')
  Example 1: R. v. Carosella (1997)
  Example

In [44]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_decision_title.items():
    pct = count / len(tokenized_decision_title)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_decision_title)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 16
  Total instances in rare patterns: 66
  Outlier mass percentage: 4.81%
  Vocabulary quality score: 95.19%


### Legislation title

In [45]:
legislation_title = get_sublabel_strings(all_annotations, 'legislation', 'title', max_items=None)

In [46]:
import re
from typing import List


def normalize_legislation_title(text: str) -> List[str]:

    # ------------------------------------------------------------------ #
    # 0. Pre-clean                                                         #
    # ------------------------------------------------------------------ #
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text).strip()

    # ------------------------------------------------------------------ #
    # 1. Fix camelCase run-together words                                  #
    #    "QuebecCharter" -> "Quebec Charter"                              #
    #    "ProfessionalCode" -> "Professional Code"                        #
    # ------------------------------------------------------------------ #
    text = re.sub(r'([a-zÀ-öø-ÿ])([A-ZÀ-Ö])', r'\1 \2', text)

    # ------------------------------------------------------------------ #
    # 2. Strip citation markers [16], [1]                                  #
    # ------------------------------------------------------------------ #
    text = re.sub(r'\[\d+\]', ' ', text)

    # ------------------------------------------------------------------ #
    # 3. Semantic substitutions — ORDER MATTERS                           #
    # ------------------------------------------------------------------ #

    _clause = r'(?:\s+(?:of|respecting|regarding|concerning|relating\s+to|du|de|des|la|the)\s+[^,\(]+?)?'
    _pre    = r'(?:(?:[\w\'\-]+\s+){0,6})'

    # -- BYLAW (before ACT so "By-law" isn't swallowed) --
    text = re.sub(
        r'\bBy[\u2011\-]?law\b' + _clause,
        'BYLAW', text, flags=re.IGNORECASE
    )

    # -- RULES OF COURT --
    text = re.sub(
        r'\b' + _pre + r'Rules?\b(?:\s+of\s+[^,\(]+?)?(?=\s*[,\(]|$)',
        'RULES', text, flags=re.IGNORECASE
    )

    # -- CHARTER --
    text = re.sub(
        r'\b' + _pre + r'Charter\b' + _clause,
        'CHARTER', text, flags=re.IGNORECASE
    )

    # -- CONSTITUTION (not followed by Act — that goes to ACT) --
    text = re.sub(
        r'\b' + _pre + r'Constitution\b(?!\s+Act)' + _clause,
        'CONSTITUTION', text, flags=re.IGNORECASE
    )

    # -- ACT / LAW (includes "Constitution Act", "Act respecting ...") --
    text = re.sub(
        r'\b' + _pre + r'(?:Act|Law)\b' + _clause,
        'ACT', text, flags=re.IGNORECASE
    )

    # -- CODE --
    text = re.sub(
        r'\b' + _pre + r'Code\b' + _clause,
        'CODE', text, flags=re.IGNORECASE
    )

    # -- REGULATION --
    text = re.sub(
        r'\b' + _pre + r'Regulations?\b' + _clause,
        'REGULATION', text, flags=re.IGNORECASE
    )

    # -- CCP (explicit, before generic ABBREV) --
    text = re.sub(r'C\.C\.P\.', 'CCP', text)

    # -- ABBREV: R.S.C.  R.S.O.  S.C.  c.p.c. --
    text = re.sub(r'(?:[A-Z]\.){2,}|(?:[A-Z]\.[a-z]+\.)+', 'ABBREV', text)

    # -- DATE: year-range 1961-62 or plain year 1982 --
    text = re.sub(r'\b\d{4}[\u2011\-]\d{2,4}\b', 'DATE', text)
    text = re.sub(r'\b(?:1[5-9]\d{2}|20\d{2})\b', 'DATE', text)

    # -- REF_ID: session/by-law numeric id 106-114  1475-92 --
    text = re.sub(r'\b\d{1,4}[\u2011\-]\d{2,4}\b', 'REF_ID', text)

    # -- NUM: bare number --
    text = re.sub(r'\b\d+\b', 'NUM', text)

    # ------------------------------------------------------------------ #
    # 4. Normalise punctuation spacing                                     #
    # ------------------------------------------------------------------ #
    text = re.sub(r'\s*,\s*', ' COMMA ', text)
    text = re.sub(r'\s*\(\s*', ' ( ', text)
    text = re.sub(r'\s*\)\s*', ' ) ', text)

    # ------------------------------------------------------------------ #
    # 5. Final clean and split                                             #
    # ------------------------------------------------------------------ #
    text = re.sub(r'[ \t]+', ' ', text).strip()

    return text.split()

In [47]:
tokenized_legislation_title = [normalize_legislation_title(x) for x in legislation_title]

for raw, tok in zip(legislation_title, tokenized_legislation_title):
    print(f"{raw:25} → {tok}")

Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter                   → ['CHARTER']
Canadian Charter of
Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter                   → ['CHARTER']
Canadian Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Canadian
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Canadian Charter of Rights
and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter                   → ['CHARTER']
Canadian Charter of Rights
and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms']
Canadian Charter of Rights and Freedoms → ['CHARTERights', 'and', 'Freedoms'

In [48]:
pattern_counts_legislation_title = Counter(tuple(seq) for seq in tokenized_legislation_title)

print("\nPatterns (Legislation Titles):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_legislation_title.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_legislation_title)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_legislation_title):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(legislation_title[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Legislation Titles):

[39.60%] ('ACT',)
  Example 1: Juries Act
  Example 2: Corporations Act
  Example 3: Extradition Act

[28.22%] ('CHARTER',)
  Example 1: Charter
  Example 2: Charter
  Example 3: Charter

[8.09%] ('CODE',)
  Example 1: Criminal Code
  Example 2: Criminal Code
  Example 3: Criminal Code

[6.60%] ('PSLRA',)
  Example 1: PSLRA
  Example 2: PSLRA
  Example 3: PSLRA

[3.47%] ('CHARTERights', 'and', 'Freedoms')
  Example 1: Charter of Rights and Freedoms
  Example 2: Canadian Charter of
Rights and Freedoms
  Example 3: Charter of Rights and Freedoms

[2.31%] ('ACT', 'COMMA', 'DATE')
  Example 1: Constitution Act, 1982
  Example 2: Constitution Act, 1982
  Example 3: Municipal
Act, 2001

[1.32%] ('FMIOA',)
  Example 1: FMIOA
  Example 2: FMIOA
  Example 3: FMIOA

[1.16%] ('Right', 'ACT')
  Example 1: Right to Information and Protection of Privacy Act
  Example 2: Right to Information and Protection of
Privacy Act
  Example 3: Right to Information and
Protectio

In [49]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_legislation_title.items():
    pct = count / len(tokenized_legislation_title)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_legislation_title)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 27
  Total instances in rare patterns: 49
  Outlier mass percentage: 8.09%
  Vocabulary quality score: 91.91%


#### Secondary source title

In [50]:
sec_source_title = get_sublabel_strings(all_annotations, 'secondary sources', 'title', max_items=None)

#### Secondary source authors

In [51]:
sec_source_authors = get_sublabel_strings(all_annotations, 'secondary sources', 'authors', max_items=None)

In [52]:
import re
from typing import List

def normalize_sec_sources_authors(text: str) -> List[str]:
    # ------------------------------------------------------------------
    # 0. Pre-clean
    # ------------------------------------------------------------------
    text = re.sub(r'[\r\n]+', ' ', text)
    text = re.sub(r'[ \t]{2,}', ' ', text).strip()

    # ------------------------------------------------------------------
    # 1. Strip titles
    # ------------------------------------------------------------------
    text = re.sub(r'\b(Professor|Prof\.|Dr\.)\s+', 'TITLE ', text, flags=re.IGNORECASE)

    # ------------------------------------------------------------------
    # 2. Strip trailing qualifiers
    # ------------------------------------------------------------------
    text = re.sub(r',?\s+with\s+the\s+assistance\s+of\s+.+$', '', text, flags=re.IGNORECASE)

    # ------------------------------------------------------------------
    # 3. et al.
    # ------------------------------------------------------------------
    text = re.sub(r'\bet\s+al\.?', 'ETAL', text, flags=re.IGNORECASE)

    # ------------------------------------------------------------------
    # 4. Tokenise into shape tokens
    # ------------------------------------------------------------------
    tokens = []
    # Split on separators while keeping them
    raw = re.split(r'(\s*,\s+and\s+|\s+and\s+|,)', text)

    for chunk in raw:
        chunk = chunk.strip()
        if not chunk:
            continue

        # Punctuation separators
        if chunk == ',':
            tokens.append(',')
            continue
        if re.fullmatch(r',?\s*(and|AND)\s*,?', chunk):
            tokens.append('AND')
            continue

        # Shape-classify each word/token in the chunk
        words = chunk.split()
        for w in words:
            tokens.append(_shape(w))

    return tokens


def _shape(w: str) -> str:
    """Map one word to its structural shape token."""

    if w == 'ETAL':
        return 'ETAL'
    if w == 'TITLE':
        return 'TITLE'

    # Single letter + optional dot → INITIAL  (must come before AUTHOR)
    if re.fullmatch(r'[A-ZÁÉÍÓÚÀÈÙÂÊÎÔÛÄËÏÖÜÇŒÆ]\.?', w):
        return 'INITIAL'

    # Compound initials: "E.A.", "J.M." etc.
    if re.fullmatch(r'([A-Z]\.){2,}', w):
        return 'INITIALS'

    # Acronym: all caps 2+ (AASHTO)
    if re.fullmatch(r'[A-Z]{2,}', w):
        return 'ACRONYM'

    # ALL-CAPS unicode word (DUMONT in "Hélène DUMONT")
    if re.fullmatch(r'[A-ZÁÉÍÓÚÀÈÙÂÊÎÔÛÄËÏÖÜÇ]{2,}', w):
        return 'AUTHOR_CAPS'

    # Any name-like word:
    #   - standard capitalised          → Dumont, Paul, Hélène
    #   - camelCase/internal caps       → DeMarco, MacDonald
    #   - hyphenated                    → Cohen-Eliya, Cohen‑Eliya (non-breaking hyphen)
    #   - accented                      → Macdonald, Jolowicz
    if re.fullmatch(
        r'[A-ZÁÉÍÓÚÀÈÙÂÊÎÔÛÄËÏÖÜÇŒÆ]'          # must start with capital
        r'[A-Za-záéíóúàèùâêîôûäëïöüçœæ]*'       # lowercase body
        r'(?:'
            r'[A-ZÁÉÍÓÚÀÈÙÂÊÎÔÛÄËÏÖÜÇŒÆ]'       # internal cap (DeMarco)
            r'[A-Za-záéíóúàèùâêîôûäëïöüçœæ]*'
        r'|'
            r'[\-\u2011\u2010]'                   # hyphen variants
            r'[A-ZÁÉÍÓÚÀÈÙÂÊÎÔÛÄËÏÖÜÇŒÆ]'
            r'[A-Za-záéíóúàèùâêîôûäëïöüçœæ]*'
        r')*',
        w
    ):
        return 'AUTHOR'

    return 'UNKNOWN'  # explicit fallback — helps you spot truly unhandled cases

In [53]:
tokenized_sec_authors = [normalize_sec_sources_authors(x) for x in sec_source_authors]

for raw, tok in zip(sec_source_authors, tokenized_sec_authors):
    print(f"{raw:25} → {tok}")

Dicey                     → ['AUTHOR']
Head, I.                  → ['AUTHOR', ',', 'INITIAL']
Petersen,
C.              → ['AUTHOR', ',', 'INITIAL']
Russell, D.               → ['AUTHOR', ',', 'INITIAL']
Schulman and
Meyers       → ['AUTHOR', 'AND', 'AUTHOR']
Ivan Head                 → ['AUTHOR', 'AUTHOR']
Dicey                     → ['AUTHOR']
Dicey                     → ['AUTHOR']
Dawn Russell              → ['AUTHOR', 'AUTHOR']
Henry Hansmann            → ['AUTHOR', 'AUTHOR']
C. Petersen               → ['INITIAL', 'AUTHOR']
Arthurs, H. W.            → ['AUTHOR', ',', 'INITIAL', 'INITIAL']
Barak, Aharon             → ['AUTHOR', ',', 'AUTHOR']
Biddulph, Michelle        → ['AUTHOR', ',', 'AUTHOR']
Bingham, Tom              → ['AUTHOR', ',', 'AUTHOR']
Brouwer, Andrew           → ['AUTHOR', ',', 'AUTHOR']
Brown, Donald J. M., and John M. Evans, with the assistance of David Fairlie → ['AUTHOR', ',', 'AUTHOR', 'INITIAL', 'INITIAL', 'AND', 'AUTHOR', 'INITIAL', 'AUTHOR']
Brownlie, Ian     

In [54]:
pattern_counts_sec_authors = Counter(tuple(seq) for seq in tokenized_sec_authors)

print("\nPatterns (Secondary Source Authors):")
print("=" * 80)
for pattern, count in sorted(pattern_counts_sec_authors.items(), key=lambda x: x[1], reverse=True):
    pct = count/len(tokenized_sec_authors)
    print(f"\n[{pct:.2%}] {pattern}")
    # Show up to 3 examples
    examples = []
    for i, seq in enumerate(tokenized_sec_authors):
        if tuple(seq) == pattern and len(examples) < 3:
            examples.append(sec_source_authors[i])
    for j, ex in enumerate(examples, 1):
        print(f"  Example {j}: {ex[:60]}{'...' if len(ex) > 60 else ''}")


Patterns (Secondary Source Authors):

[16.82%] ('AUTHOR',)
  Example 1: Dicey
  Example 2: Dicey
  Example 3: Dicey

[15.45%] ('AUTHOR', ',', 'AUTHOR')
  Example 1: Barak, Aharon
  Example 2: Biddulph, Michelle
  Example 3: Bingham, Tom

[14.55%] ('AUTHOR', 'AUTHOR')
  Example 1: Ivan Head
  Example 2: Dawn Russell
  Example 3: Henry Hansmann

[8.18%] ('AUTHOR', ',', 'AUTHOR', 'INITIAL')
  Example 1: Coady, Jonathan M
  Example 2: Cromwell, Thomas A
  Example 3: DeMarco, Jerry V

[6.36%] ('AUTHOR', 'INITIAL', 'AUTHOR')
  Example 1: John A. Willes
  Example 2: Kevin M. Stack
  Example 3: David J. Mullan

[5.91%] ('INITIAL', 'AUTHOR')
  Example 1: C. Petersen
  Example 2: P. Daly
  Example 3: B. McLachlin

[4.55%] ('AUTHOR', 'AND', 'AUTHOR')
  Example 1: Schulman and
Meyers
  Example 2: Grant and Sossin
  Example 3: Wade and Forsyth

[4.09%] ('INITIAL', 'INITIAL', 'AUTHOR')
  Example 1: D. J. Mullan
  Example 2: J. T. Robertson
  Example 3: J. T. Robertson

[1.36%] ('AUTHOR', ',', 'INIT

In [55]:
# Calculate outlier mass (patterns < 1%)
outlier_count = 0
outlier_mass = 0
for pattern, count in pattern_counts_sec_authors.items():
    pct = count / len(tokenized_sec_authors)
    if pct < 0.01:
        outlier_count += 1
        outlier_mass += count

outlier_mass_pct = (outlier_mass / len(tokenized_sec_authors)) * 100
print(f"\n{'='*80}")
print(f"Outlier Mass Analysis (< 1%):")
print(f"  Number of unique patterns: {outlier_count}")
print(f"  Total instances in rare patterns: {outlier_mass}")
print(f"  Outlier mass percentage: {outlier_mass_pct:.2f}%")
print(f"  Vocabulary quality score: {100 - outlier_mass_pct:.2f}%")
print(f"{'='*80}")


Outlier Mass Analysis (< 1%):
  Number of unique patterns: 31
  Total instances in rare patterns: 41
  Outlier mass percentage: 18.64%
  Vocabulary quality score: 81.36%
